# Day 1: Multi-Agent AI Systems - Foundations & Incident Triage

## Advanced Multi-Agent AI Systems Training
### For Support & Meta Engineers

---

## 🎯 Learning Objectives

By the end of this session, you will:
1. Understand multi-agent architecture patterns
2. Implement agent communication & coordination
3. Build a production-ready incident triage system
4. Apply best practices for safe automation

## 📋 Session Outline

1. **Introduction to Multi-Agent Systems** (30 min)
2. **LLM Infrastructure & MockLLM** (20 min)
3. **Agent Architecture & Communication** (40 min)
4. **Building the Triage System** (60 min)
5. **Hands-on Exercises** (50 min)

---

## Part 1: Setup & Introduction

### 1.1 Environment Setup

In [ ]:
import sys
import os
import json
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✓ Project root: {project_root}")
print(f"✓ Python version: {sys.version.split()[0]}")
print(f"✓ Working directory: {os.getcwd()}")

### 1.2 Initialize LLM (MockLLM or OpenAI)

This course supports **two modes**:
- **MockLLM** (default): Runs without API keys, deterministic responses
- **OpenAI** (optional): Real LLM if `OPENAI_API_KEY` is set in `.env`

The system automatically detects which mode to use.

In [ ]:
from src.llm import get_llm, print_llm_stats

llm = get_llm(force_mock=False, deterministic=True, verbose=True)

print("\n" + "="*60)
print("Testing LLM...")
print("="*60)

test_response = llm.generate(
    "Classify this incident: Database connection timeout in production",
    temperature=0.3,
    max_tokens=200
)

print(f"\nResponse: {test_response['response'][:200]}...")
print(f"Model: {test_response['model']}")
print(f"Tokens: {test_response['tokens']}")
print(f"Cost: ${test_response['cost']:.4f}")

---

## Part 2: Multi-Agent System Fundamentals

### 2.1 What is a Multi-Agent System?

A **multi-agent system** consists of:
- **Multiple autonomous agents** with specialized roles
- **Communication mechanisms** for information exchange
- **Coordination patterns** for collaborative problem-solving
- **Shared context** for maintaining state

### 2.2 Why Multi-Agent for Incident Management?

**Single-agent limitations:**
- ❌ Tries to do everything → poor at specialization
- ❌ Long prompts → context overflow
- ❌ No parallelization → slow processing
- ❌ Hard to debug and maintain

**Multi-agent advantages:**
- ✅ Specialized expertise per agent
- ✅ Modular, testable components
- ✅ Parallel processing capability
- ✅ Clear separation of concerns
- ✅ Easier to add/remove capabilities

### 2.3 Agent Architecture

Let's explore the base `Agent` class:

In [ ]:
from src.agents import Agent, AgentRole

classifier_agent = Agent(
    name="IncidentClassifier",
    role=AgentRole.CLASSIFIER,
    llm=llm,
    system_prompt="You are an expert incident classifier. Analyze incidents and assign severity (P0-P4) and category."
)

print(f"Agent created: {classifier_agent}")
print(f"Role: {classifier_agent.role.value}")
print(f"System Prompt: {classifier_agent.system_prompt[:100]}...")

### 2.4 Agent Processing

Let's see how an agent processes input:

In [ ]:
incident_data = {
    "title": "Database Connection Pool Exhausted",
    "description": "Production database experiencing connection pool exhaustion. Users unable to access application.",
    "affected_users": 1500
}

result = classifier_agent.process(incident_data)

print("\n" + "="*60)
print("Classification Result:")
print("="*60)
print(json.dumps(result, indent=2))

---

## Part 3: Agent Communication

### 3.1 Message Passing

Agents communicate via a **MessageBus** - a centralized message broker.

In [ ]:
from src.agents import Message, MessageBus, MessagePriority

message_bus = MessageBus()

msg = Message(
    sender="ClassifierAgent",
    receiver="RouterAgent",
    content={"severity": "P0", "category": "Database"},
    message_type="classification_result",
    priority=MessagePriority.CRITICAL
)

message_id = message_bus.send(msg)
print(f"✓ Message sent: {message_id}")
print(f"  From: {msg.sender}")
print(f"  To: {msg.receiver}")
print(f"  Priority: {msg.priority.name}")

received = message_bus.receive("RouterAgent")
print(f"\n✓ RouterAgent received {len(received)} message(s)")

### 3.2 Message Flow Visualization

In [ ]:
message_bus.send(Message("Agent1", "Agent2", {"data": "test1"}, priority=MessagePriority.HIGH))
message_bus.send(Message("Agent2", "Agent3", {"data": "test2"}, priority=MessagePriority.NORMAL))
message_bus.send(Message("Agent3", "Agent1", {"data": "test3"}, priority=MessagePriority.CRITICAL))

print(message_bus.visualize_flow())

stats = message_bus.get_stats()
print("\nMessage Bus Statistics:")
print(json.dumps(stats, indent=2))

---

## Part 4: Building the Incident Triage System

### 4.1 System Architecture

Our triage system has **3 specialized agents**:

1. **Classifier Agent**: Assigns severity (P0-P4) and category
2. **Deduplication Agent**: Finds similar/duplicate incidents
3. **Router Agent**: Routes to appropriate team

```
Incident → Classifier → Deduplicator → Router → Team Assignment
```

### 4.2 Load Sample Incidents

In [ ]:
with open('data/sample_incidents.json', 'r') as f:
    incidents = json.load(f)

print(f"✓ Loaded {len(incidents)} sample incidents\n")

for i, incident in enumerate(incidents[:3], 1):
    print(f"{i}. {incident['id']}: {incident['title']}")
    print(f"   Severity: {incident['severity']}, Users: {incident['affected_users']}")
    print()

### 4.3 Create Triage Agents

In [ ]:
llm_triage = get_llm(force_mock=False, deterministic=True, verbose=False)

classifier = Agent(
    name="IncidentClassifier",
    role=AgentRole.CLASSIFIER,
    llm=llm_triage
)

deduplicator = Agent(
    name="IncidentDeduplicator",
    role=AgentRole.DEDUPLICATOR,
    llm=llm_triage
)

router = Agent(
    name="IncidentRouter",
    role=AgentRole.ROUTER,
    llm=llm_triage
)

print("✓ Created 3 specialized agents:")
print(f"  1. {classifier}")
print(f"  2. {deduplicator}")
print(f"  3. {router}")

### 4.4 Orchestrate the Workflow

We'll use the `Orchestrator` to coordinate the agents:

In [ ]:
from src.agents import Orchestrator, WorkflowStep

orchestrator = Orchestrator(name="IncidentTriageWorkflow")

orchestrator.register_agent("classifier", classifier)
orchestrator.register_agent("deduplicator", deduplicator)
orchestrator.register_agent("router", router)

workflow = [
    WorkflowStep(name="classify", agent_name="classifier"),
    WorkflowStep(name="deduplicate", agent_name="deduplicator", depends_on=["classify"]),
    WorkflowStep(name="route", agent_name="router", depends_on=["classify", "deduplicate"])
]

orchestrator.build_workflow(workflow)

print(orchestrator.visualize_workflow())

### 4.5 Execute Triage Workflow

In [ ]:
test_incident = incidents[0]

print("\n" + "="*60)
print(f"Processing Incident: {test_incident['id']}")
print(f"Title: {test_incident['title']}")
print("="*60 + "\n")

workflow_result = orchestrator.execute_workflow(test_incident, verbose=True)

print("\n" + "="*60)
print("Workflow Results:")
print("="*60)
print(f"Status: {workflow_result['status']}")
print(f"Duration: {workflow_result['duration_seconds']:.2f}s")
print(f"Steps Executed: {workflow_result['steps_executed']}")

### 4.6 Examine Individual Results

In [ ]:
print("\n📊 CLASSIFICATION RESULT:")
print("="*60)
classify_result = orchestrator.get_step_result("classify")
print(classify_result['response'])

print("\n📊 DEDUPLICATION RESULT:")
print("="*60)
dedup_result = orchestrator.get_step_result("deduplicate")
print(dedup_result['response'])

print("\n📊 ROUTING RESULT:")
print("="*60)
route_result = orchestrator.get_step_result("route")
print(route_result['response'])

### 4.7 Process Multiple Incidents

In [ ]:
print("\n" + "="*60)
print("Processing Multiple Incidents")
print("="*60 + "\n")

results_summary = []

for incident in incidents[:5]:
    orchestrator.reset()
    
    print(f"\n▶️  Processing: {incident['id']} - {incident['title'][:50]}...")
    
    result = orchestrator.execute_workflow(incident, verbose=False)
    
    results_summary.append({
        'incident_id': incident['id'],
        'title': incident['title'],
        'status': result['status'],
        'duration': result['duration_seconds']
    })
    
    print(f"   ✓ Status: {result['status']}, Duration: {result['duration_seconds']:.2f}s")

print("\n" + "="*60)
print("Summary:")
print("="*60)
for summary in results_summary:
    print(f"  {summary['incident_id']}: {summary['status']} ({summary['duration']:.2f}s)")

---

## Part 5: Advanced Features

### 5.1 Agent State Management

In [ ]:
classifier.update_state('incidents_processed', 5)
classifier.update_state('p0_count', 2)
classifier.update_state('p1_count', 3)

print("Agent State:")
print(f"  Incidents Processed: {classifier.get_state('incidents_processed')}")
print(f"  P0 Count: {classifier.get_state('p0_count')}")
print(f"  P1 Count: {classifier.get_state('p1_count')}")

history = classifier.get_history(limit=3)
print(f"\nExecution History: {len(history)} recent executions")
for i, exec_record in enumerate(history, 1):
    print(f"  {i}. {exec_record['timestamp']}: {exec_record['duration_ms']:.2f}ms")

### 5.2 Conditional Workflow Steps

Add conditional logic to workflows:

In [ ]:
def should_escalate(context):
    classify_result = context.get('classify_result', {})
    response = classify_result.get('response', '')
    return 'P0' in response or 'P1' in response

orchestrator_conditional = Orchestrator(name="ConditionalTriageWorkflow")
orchestrator_conditional.register_agent("classifier", classifier)
orchestrator_conditional.register_agent("router", router)

conditional_workflow = [
    WorkflowStep(name="classify", agent_name="classifier"),
    WorkflowStep(
        name="escalate",
        agent_name="router",
        depends_on=["classify"],
        condition=should_escalate
    )
]

orchestrator_conditional.build_workflow(conditional_workflow)

print("Conditional Workflow:")
print(orchestrator_conditional.visualize_workflow())

p0_incident = incidents[0]
result = orchestrator_conditional.execute_workflow(p0_incident, verbose=True)

### 5.3 LLM Usage Statistics

In [ ]:
print_llm_stats(llm_triage)

---

## Part 6: Hands-On Exercises

### Exercise 1: Add a Summarizer Agent

**Task**: Create a new agent that summarizes the triage results.

**Requirements**:
- Role: `AgentRole.SUMMARIZER`
- Input: Results from all three agents
- Output: Concise summary with key decisions

**Hint**: Add it as the final step in the workflow.

In [ ]:
summarizer = Agent(
    name="IncidentSummarizer",
    role=AgentRole.SUMMARIZER,
    llm=llm_triage
)

orchestrator_ex1 = Orchestrator(name="TriageWithSummary")
orchestrator_ex1.register_agent("classifier", classifier)
orchestrator_ex1.register_agent("deduplicator", deduplicator)
orchestrator_ex1.register_agent("router", router)
orchestrator_ex1.register_agent("summarizer", summarizer)

workflow_ex1 = [
    WorkflowStep(name="classify", agent_name="classifier"),
    WorkflowStep(name="deduplicate", agent_name="deduplicator", depends_on=["classify"]),
    WorkflowStep(name="route", agent_name="router", depends_on=["classify", "deduplicate"]),
    WorkflowStep(name="summarize", agent_name="summarizer", depends_on=["classify", "deduplicate", "route"])
]

orchestrator_ex1.build_workflow(workflow_ex1)

result_ex1 = orchestrator_ex1.execute_workflow(incidents[1], verbose=True)

print("\n📝 SUMMARY:")
print("="*60)
summary_result = orchestrator_ex1.get_step_result("summarize")
print(summary_result['response'])

### Exercise 2: Priority-Based Routing

**Task**: Modify the router agent to use different teams based on severity.

**Requirements**:
- P0/P1: Route to "Critical Response Team"
- P2/P3: Route to "Standard Support Team"
- P4: Route to "Maintenance Team"

**Hint**: Use custom system prompts or post-processing.

In [ ]:
priority_router = Agent(
    name="PriorityRouter",
    role=AgentRole.ROUTER,
    llm=llm_triage,
    system_prompt="""You are an expert incident router. Route incidents based on severity:
    - P0/P1: Critical Response Team (24/7 on-call)
    - P2/P3: Standard Support Team (business hours)
    - P4: Maintenance Team (scheduled work)
    
    Always specify the team, SLA, and escalation requirements."""
)

orchestrator_ex2 = Orchestrator(name="PriorityBasedTriage")
orchestrator_ex2.register_agent("classifier", classifier)
orchestrator_ex2.register_agent("router", priority_router)

workflow_ex2 = [
    WorkflowStep(name="classify", agent_name="classifier"),
    WorkflowStep(name="route", agent_name="router", depends_on=["classify"])
]

orchestrator_ex2.build_workflow(workflow_ex2)

print("Testing Priority-Based Routing:\n")
for incident in incidents[:3]:
    orchestrator_ex2.reset()
    print(f"\n▶️  {incident['id']}: {incident['title'][:40]}...")
    result = orchestrator_ex2.execute_workflow(incident, verbose=False)
    route_result = orchestrator_ex2.get_step_result("route")
    print(f"   {route_result['response'][:150]}...")

### Exercise 3: Batch Processing with Metrics

**Task**: Process all incidents and collect performance metrics.

**Requirements**:
- Process all 10 sample incidents
- Track: total time, avg time per incident, success rate
- Generate a summary report

**Hint**: Use the `MetricsCollector` utility.

In [ ]:
from src.utils import MetricsCollector
from datetime import datetime

metrics = MetricsCollector()
metric = metrics.start_tracking("BatchIncidentProcessing")

batch_results = []
success_count = 0
failed_count = 0

print("\n" + "="*60)
print("Batch Processing All Incidents")
print("="*60 + "\n")

for i, incident in enumerate(incidents, 1):
    orchestrator.reset()
    
    start = datetime.now()
    result = orchestrator.execute_workflow(incident, verbose=False)
    duration = (datetime.now() - start).total_seconds()
    
    metric.record_timing("incident_processing", duration * 1000)
    
    if result['status'] == 'COMPLETED':
        success_count += 1
        metric.increment_counter('successful_incidents')
    else:
        failed_count += 1
        metric.increment_counter('failed_incidents')
    
    batch_results.append({
        'incident_id': incident['id'],
        'status': result['status'],
        'duration': duration
    })
    
    print(f"{i:2d}. {incident['id']}: {result['status']:10s} ({duration:.2f}s)")

metric.record_metric('total_incidents', len(incidents))
metric.record_metric('success_rate', success_count / len(incidents))
metrics.stop_tracking()

print("\n" + "="*60)
print("Batch Processing Summary")
print("="*60)
print(f"Total Incidents: {len(incidents)}")
print(f"Successful: {success_count}")
print(f"Failed: {failed_count}")
print(f"Success Rate: {success_count/len(incidents)*100:.1f}%")

metrics.print_summary()

---

## Part 7: Production Considerations

### 7.1 Error Handling & Retries

The orchestrator supports automatic retries:

In [ ]:
workflow_with_retries = [
    WorkflowStep(name="classify", agent_name="classifier", max_retries=3),
    WorkflowStep(name="deduplicate", agent_name="deduplicator", depends_on=["classify"], max_retries=2),
    WorkflowStep(name="route", agent_name="router", depends_on=["classify"], max_retries=2)
]

print("Workflow with Retry Configuration:")
for step in workflow_with_retries:
    print(f"  - {step.name}: max_retries={step.max_retries}")

### 7.2 Guardrails & Safety

**Key safety principles**:
1. **Validation**: Always validate agent outputs
2. **Human-in-the-loop**: Critical decisions require approval
3. **Audit trails**: Log all decisions and reasoning
4. **Rollback capability**: Be able to undo automated actions
5. **Rate limiting**: Prevent runaway automation

In [ ]:
def validate_classification(result):
    response = result.get('response', '')
    
    valid_severities = ['P0', 'P1', 'P2', 'P3', 'P4']
    has_severity = any(sev in response for sev in valid_severities)
    
    if not has_severity:
        print("⚠️  WARNING: Classification missing severity level")
        return False
    
    print("✓ Classification validation passed")
    return True

test_result = classifier.process(incidents[0])
is_valid = validate_classification(test_result)

### 7.3 Monitoring & Observability

In [ ]:
summary = orchestrator.get_workflow_summary()

print("Workflow Observability:")
print(json.dumps(summary, indent=2))

print("\nAgent Execution History:")
for agent_name, agent in orchestrator.agents.items():
    history = agent.get_history()
    print(f"  {agent_name}: {len(history)} executions")
    if history:
        avg_duration = sum(h['duration_ms'] for h in history) / len(history)
        print(f"    Avg duration: {avg_duration:.2f}ms")

---

## Part 8: Key Takeaways

### What We Learned Today:

1. ✅ **Multi-agent architecture** beats single-agent for complex tasks
2. ✅ **Specialized agents** with clear roles improve accuracy
3. ✅ **Orchestration** enables complex workflows with dependencies
4. ✅ **Message passing** provides flexible agent communication
5. ✅ **MockLLM** enables testing without API costs
6. ✅ **Guardrails** are essential for production safety

### Production Checklist:

- [ ] Validate all agent outputs
- [ ] Implement retry logic for failures
- [ ] Add human approval for critical actions
- [ ] Log all decisions with reasoning
- [ ] Monitor performance metrics
- [ ] Test with MockLLM before deploying
- [ ] Set up alerting for anomalies
- [ ] Document agent responsibilities

### Tomorrow's Preview (Day 2):

- **Hierarchical agent systems** (coordinator + workers)
- **Evidence-based reasoning** for RCA
- **5-agent root cause analysis** system
- **Advanced guardrails** and safety patterns
- **Production deployment** strategies

---

## 🎉 Day 1 Complete!

Great work! You've built a production-ready incident triage system with multiple specialized agents.

**Next Steps**:
1. Experiment with different agent configurations
2. Try the exercises with your own incident data
3. Test with real LLMs (optional) by setting up `.env`
4. Review the code in `src/` for deeper understanding

See you tomorrow for Day 2! 🚀